In [ ]:
from pylab import *

In [ ]:
plt.rcParams.update({'font.size': 12})

# Resolution Error

We will first estimate the error from the resolution $h = \Delta \rho = \Delta z$. We should expect it to be fourth order, since we set all derivatives to fourth order discretization.

We will examine eight different resolutions, all of which will extend to $r_\infty = 128$. Thus, we have an array for the number of interior points and an array for the resolutions.

In [ ]:
res_Ns = np.linspace(128, 1024, 8).astype(int)
print(res_Ns)

In [ ]:
res_hs = 128.0 / res_Ns
print(res_hs)

List the output directories with our datasets.

In [ ]:
!ls Resolution

From this, generate an array with the directory names.

In [ ]:
res_dirnames = ['Resolution/l=6,w=0.70000,dr=1.000,N=0128/',
            'Resolution/l=6,w=0.70000,dr=0.500,N=0256/',
            'Resolution/l=6,w=0.70000,dr=0.333,N=0384/',
            'Resolution/l=6,w=0.70000,dr=0.250,N=0512/',
            'Resolution/l=6,w=0.70000,dr=0.200,N=0640/',
            'Resolution/l=6,w=0.70000,dr=0.166,N=0768/',
            'Resolution/l=6,w=0.70000,dr=0.143,N=0896/',
            'Resolution/l=6,w=0.70000,dr=0.125,N=1024/']

Our main quantities are the Komar masses and angular momenta which we extract from the datasets.

In [ ]:
res_M_K1 = np.zeros(len(res_dirnames))
res_M_K2 = np.zeros(len(res_dirnames))
res_J_K1 = np.zeros(len(res_dirnames))
res_J_K2 = np.zeros(len(res_dirnames))
res_psi0 = np.zeros(len(res_dirnames))

for i in range(len(res_dirnames)):
    res_M_K1[i] = np.genfromtxt(res_dirnames[i] + "/M_Komar1.asc")[-1]
    res_M_K2[i] = np.genfromtxt(res_dirnames[i] + "/M_Komar2.asc")[-1]
    res_J_K1[i] = np.genfromtxt(res_dirnames[i] + "/J_Komar1.asc")[-1]
    res_J_K2[i] = np.genfromtxt(res_dirnames[i] + "/J_Komar2.asc")[-1]
    res_psi0[i] = np.genfromtxt(res_dirnames[i] + "/sph_psi_f.asc", usecols = 0)[0]

In [ ]:
res_M_K1[-1], res_J_K1[-1]

We now calculate the "self-convergence" from the expression

$$c(n) = \lim_{h\to 0}\,\frac{|u(h_1) - u(h_2)|}{|u(h_2) - u(h_3)|} = \frac{h_1^n - h_2^n}{h_2^n - h_3^n}\,.$$

Fitting our relative errors with the above curve with $n = 4$ we obtain the following figure.

In [ ]:
fig, ax = plt.subplots(figsize = (8, 6))

ax.plot(-np.log10(res_hs[1:]), np.log10(np.abs((res_hs[1:]**4 - res_hs[:-1]**4))), label = r"$c(4)$")

ax.plot(-np.log10(res_hs[1:]), np.log10(np.abs(res_M_K1[1:] - res_M_K1[:-1])), "-*", label = r"$M_{Komar1}$")
ax.plot(-np.log10(res_hs[1:]), np.log10(np.abs(res_M_K2[1:] - res_M_K2[:-1])), "-*", label = r"$M_{Komar2}$")
ax.plot(-np.log10(res_hs[1:]), np.log10(np.abs(res_J_K1[1:] - res_J_K1[:-1])), "-*", label = r"$J_{Komar1}$")
ax.plot(-np.log10(res_hs[1:]), np.log10(np.abs(res_J_K2[1:] - res_J_K2[:-1])), "-*", label = r"$J_{Komar2}$")
ax.plot(-np.log10(res_hs[1:]), np.log10(np.abs(res_psi0[1:] - res_psi0[:-1])), "-*", label = r"$\psi_0$")

ax.legend()
ax.set_xlabel(r"$\log_{10}\,1/h$")
ax.set_ylabel(r"$\log_{10}\,\delta$")
ax.set_title("Komar Mass and Angular Momentum Self-Convergence")

plt.show()

With the exception of two points at low resolution (the two volume integrals, in fact) we see that the fit is very well adjusted to $n = 4$.

Alternatively, we can approximate that our step ratios will turn out constant as $h\to 0$, in which case

$$r^n = \left(\frac{h_2}{h_3}\right)^n = c(n)\,.$$

This allows an easier fit since the equation $y^n = x$ gives $n = \tfrac{\log x}{\log y}$.

In [ ]:
r = res_hs[:-2] / res_hs[1:-1]

In [ ]:
fig, ax = plt.subplots(figsize = (8, 6))

ax.plot(-np.log10(res_hs[1:-1]), np.log10(np.abs(res_M_K1[1:-1] - res_M_K1[:-2]) / np.abs(res_M_K1[2:] - res_M_K1[1:-1])) / np.log10(r), ".", label = r"$M_{K1}$")
ax.plot(-np.log10(res_hs[1:-1]), np.log10(np.abs(res_M_K2[1:-1] - res_M_K2[:-2]) / np.abs(res_M_K2[2:] - res_M_K2[1:-1])) / np.log10(r), ".", label = r"$M_{K2}$")

ax.plot(-np.log10(res_hs[1:-1]), np.log10(np.abs(res_J_K1[1:-1] - res_J_K1[:-2]) / np.abs(res_J_K1[2:] - res_J_K1[1:-1])) / np.log10(r), ".", label = r"$J_{K1}$")
ax.plot(-np.log10(res_hs[1:-1]), np.log10(np.abs(res_J_K2[1:-1] - res_J_K2[:-2]) / np.abs(res_J_K2[2:] - res_J_K2[1:-1])) / np.log10(r), ".", label = r"$J_{K2}$")

ax.plot(-np.log10(res_hs[1:-1]), 4 * np.ones_like(res_hs[1:-1]), "--", label = r"$n = 4$")

ax.set_ylim(1, 8)
ax.legend()
ax.set_xlabel(r"$\log_{10}\,1/h$")
ax.set_ylabel(r"$n$")
ax.set_title("Komar Mass and Angular Momentum Self-Convergence")

plt.show()

Since all points are above the $n = 4$ line (which is actually an overestimate from our approximation), we can see that **we do have indeed fourth order convergence**. This allows us to estimate the error from this finite difference scheme using Richardson's extrapolation:

$$\epsilon(h_2) = \frac{1}{(h_1 / h_2)^n - 1}\,\left(u(h_1) - u(h_2)\right)\,.$$

In [ ]:
res_e_M_K1 = (res_M_K1[-2] - res_M_K1[-1]) / ((res_hs[-2] / res_hs[-1])**4 - 1.0)
res_e_M_K2 = (res_M_K2[-2] - res_M_K2[-1]) / ((res_hs[-2] / res_hs[-1])**4 - 1.0)
res_e_J_K1 = (res_J_K1[-2] - res_J_K1[-1]) / ((res_hs[-2] / res_hs[-1])**4 - 1.0)
res_e_J_K2 = (res_J_K2[-2] - res_J_K2[-1]) / ((res_hs[-2] / res_hs[-1])**4 - 1.0)
res_e_psi0 = (res_psi0[-2] - res_psi0[-1]) / ((res_hs[-2] / res_hs[-1])**4 - 1.0)

In [ ]:
print(res_M_K1[-1], res_M_K2[-1], res_J_K1[-1], res_J_K2[-1])

In [ ]:
print(res_e_M_K1, res_e_M_K2, res_e_J_K1, res_e_J_K2)

In [ ]:
print(res_e_M_K1 / res_M_K1[-1], res_e_M_K2 / res_M_K2[-1], res_e_J_K1 / res_J_K1[-1], res_e_J_K2 / res_J_K2[-1])

This allows us to calculate "better" expressions for the masses and angular momenta.

In [ ]:
res_f_M_K1 = res_M_K1[-1] - res_e_M_K1
res_f_M_K2 = res_M_K2[-1] - res_e_M_K2
res_f_J_K1 = res_J_K1[-1] - res_e_J_K1
res_f_J_K2 = res_J_K2[-1] - res_e_J_K2
res_f_psi0 = res_psi0[-1] - res_e_psi0

Using these quantities we can present another set of graphs which again prove fourth order convergence in perhaps a more familiar way.

In [ ]:
fig, ax = plt.subplots(figsize = (8, 6))

ax.plot(-np.log10(res_hs), np.log10(np.abs(res_M_K1 - res_f_M_K1)), "-*", label = r"S($M_{Komar}$)")
ax.plot(-np.log10(res_hs), np.log10(np.abs(res_M_K2 - res_f_M_K2)), "-*", label = r"V($M_{Komar}$)")
ax.plot(-np.log10(res_hs), np.log10(np.abs(res_J_K1 - res_f_J_K1)), "-*", label = r"S($J_{Komar}$)")
ax.plot(-np.log10(res_hs), np.log10(np.abs(res_J_K2 - res_f_J_K2)), "-*", label = r"V($J_{Komar}$)")
#ax.plot(-np.log10(res_hs), np.log10(np.abs(res_psi0 - res_f_psi0)), "-*", label = r"$\psi_0$")
ax.plot(-np.log10(res_hs), 4.0 * np.log10(res_hs) - 1, "-", label = r"$\mathcal{O}((\Delta\rho)^4)$")

ax.legend()
ax.set_xlabel(r"$\log_{10}\,1/\Delta\rho$")
ax.set_ylabel(r"$\log_{10} \epsilon$")
ax.set_title("Komar Mass and Angular Momentum Resolution Errors")

plt.show()

In [ ]:
fig, ax = plt.subplots(figsize = (8, 6))

ax.plot(-np.log10(res_hs), np.log10(np.abs(res_psi0 - res_f_psi0)), "-*", label = r"$\psi_0$")
ax.plot(-np.log10(res_hs), 4.0 * np.log10(res_hs) - 7, "-", label = r"$\mathcal{O}((\Delta\rho)^4)$")

ax.legend()
ax.set_xlabel(r"$\log_{10}\,1/\Delta\rho$")
ax.set_ylabel(r"$\log_{10} \epsilon$")
ax.set_title("Scalar Field Value at Origin $\psi_0$ Resolution Errors")

plt.show()

# Boundary Error

Above we saw that our discretization is indeed working to fourth order and gives relative erros at the order of $10^{-4}\%$. However, we should still study the boundary error coming from the fact that our boundary conditions are not set at spatial infinity, but rather at the last grid point called $r_\infty$. Recall that at this boundary we are establishing Robin-type boundary conditions for a function that decays as $u\to u_\infty + u_n / r^n$:

$$r\,\frac{\partial u}{\partial r} + n\,\left(u - u_\infty\right) = \mathcal{O}\left(r_\infty^{-(n+1)}\right)\,.$$



We have various functions with different $n$'s. The lapse and metric functions are $n = 1$, the shift is $n = 3$, and $\psi\,e^{+\chi r}$ is $n = l + 1$. There is no easy or direct answer as to what type of behavior to expect for the error, so we must make a full analysis. We consider sixteen simulations all with fixed resolution $h = \Delta \rho = \Delta z = 0.125$.

In [ ]:
!ls Boundary2 -C1

In [ ]:
bdy_Ns = np.array([160, 208, 271, 354, 462, 602, 785, 1024], dtype=int)

In [ ]:
bdy_rr_infs = bdy_Ns * 0.125
print(bdy_rr_infs)

Naïvely, we could expect that the error is $r_\infty^{-2}$.

In [ ]:
bdy_rr_infs**-2

In [ ]:
bdy_dirnames = ['Boundary2/l=6,w=0.70000,dr=0.125,N=0160/',
'Boundary2/l=6,w=0.70000,dr=0.125,N=0208/',
'Boundary2/l=6,w=0.70000,dr=0.125,N=0271/',
'Boundary2/l=6,w=0.70000,dr=0.125,N=0354/',
'Boundary2/l=6,w=0.70000,dr=0.125,N=0462/',
'Boundary2/l=6,w=0.70000,dr=0.125,N=0602/',
'Boundary2/l=6,w=0.70000,dr=0.125,N=0785/',
'Boundary2/l=6,w=0.70000,dr=0.125,N=1024/']


In [ ]:
bdy_M_K1 = np.zeros(len(bdy_dirnames))
bdy_M_K2 = np.zeros(len(bdy_dirnames))
bdy_J_K1 = np.zeros(len(bdy_dirnames))
bdy_J_K2 = np.zeros(len(bdy_dirnames))
bdy_w = np.zeros(len(bdy_dirnames))

for i in range(len(bdy_dirnames)):
    bdy_M_K1[i] = np.genfromtxt(bdy_dirnames[i] + "M_Komar1.asc")[-1]
    bdy_M_K2[i] = np.genfromtxt(bdy_dirnames[i] + "M_Komar2.asc")[-1]
    bdy_J_K1[i] = np.genfromtxt(bdy_dirnames[i] + "J_Komar1.asc")[-1]
    bdy_J_K2[i] = np.genfromtxt(bdy_dirnames[i] + "J_Komar2.asc")[-1]
    bdy_w[i] = np.genfromtxt(bdy_dirnames[i] + "w_f.asc").item()

In [ ]:
bdy_r = np.average(bdy_rr_infs[1:] / bdy_rr_infs[:-1])
print(bdy_r)

In [ ]:
bdy_w

In [ ]:
fig, ax = plt.subplots(figsize = (8, 6))

ax.plot(np.log10(bdy_rr_infs[1:-1]), 2.0 * np.ones_like(bdy_rr_infs[1:-1]), label = r"$n = 2$")
ax.plot(np.log10(bdy_rr_infs[1:-1]), 3.0 * np.ones_like(bdy_rr_infs[1:-1]), label = r"$n = 3$")
ax.plot(np.log10(bdy_rr_infs[1:-1]), 4.0 * np.ones_like(bdy_rr_infs[1:-1]), label = r"$n = 4$")

ax.plot(np.log10(bdy_rr_infs[1:-1]), np.log10(np.abs(bdy_M_K1[1:-1] - bdy_M_K1[:-2]) / np.abs(bdy_M_K1[2:] - bdy_M_K1[1:-1])) / np.log10(bdy_r), ".", label = r"$M_{K1}$")
ax.plot(np.log10(bdy_rr_infs[1:-1]), np.log10(np.abs(bdy_M_K2[1:-1] - bdy_M_K2[:-2]) / np.abs(bdy_M_K2[2:] - bdy_M_K2[1:-1])) / np.log10(bdy_r), ".", label = r"$M_{K2}$")
ax.plot(np.log10(bdy_rr_infs[1:-1]), np.log10(np.abs(bdy_J_K1[1:-1] - bdy_J_K1[:-2]) / np.abs(bdy_J_K1[2:] - bdy_J_K1[1:-1])) / np.log10(bdy_r), ".", label = r"$J_{K1}$")
ax.plot(np.log10(bdy_rr_infs[1:-1]), np.log10(np.abs(bdy_J_K2[1:-1] - bdy_J_K2[:-2]) / np.abs(bdy_J_K2[2:] - bdy_J_K2[1:-1])) / np.log10(bdy_r), ".", label = r"$J_{K2}$")
#ax.plot(np.log10(bdy_rr_infs[1:-1]), np.log10(np.abs(bdy_w[1:-1] - bdy_w[:-2]) / np.abs(bdy_w[2:] - bdy_w[1:-1])) / np.log10(bdy_r), ".", label = r"$\omega$")

ax.legend()
#ax.set_ylim(0,6)
ax.set_xlabel(r"$\log_{10}\,r_\infty$")
ax.set_ylabel(r"$\Delta$")
ax.set_title("Komar Mass and Angular Momentum Self-Convergence")

plt.show()

In [ ]:
np.average(np.array([np.log10(np.abs(bdy_M_K1[1:-1] - bdy_M_K1[:-2]) / np.abs(bdy_M_K1[2:] - bdy_M_K1[1:-1])) / log10(bdy_r),
np.log10(np.abs(bdy_M_K2[1:-1] - bdy_M_K2[:-2]) / np.abs(bdy_M_K2[2:] - bdy_M_K2[1:-1])) / log10(bdy_r), 
np.log10(np.abs(bdy_J_K1[1:-1] - bdy_J_K1[:-2]) / np.abs(bdy_J_K1[2:] - bdy_J_K1[1:-1])) / log10(bdy_r), 
np.log10(np.abs(bdy_J_K2[1:-1] - bdy_J_K2[:-2]) / np.abs(bdy_J_K2[2:] - bdy_J_K2[1:-1])) / log10(bdy_r)]), axis=-1),np.std(np.array([np.log10(np.abs(bdy_M_K1[1:-1] - bdy_M_K1[:-2]) / np.abs(bdy_M_K1[2:] - bdy_M_K1[1:-1])) / log10(bdy_r),
np.log10(np.abs(bdy_M_K2[1:-1] - bdy_M_K2[:-2]) / np.abs(bdy_M_K2[2:] - bdy_M_K2[1:-1])) / log10(bdy_r), 
np.log10(np.abs(bdy_J_K1[1:-1] - bdy_J_K1[:-2]) / np.abs(bdy_J_K1[2:] - bdy_J_K1[1:-1])) / log10(bdy_r), 
np.log10(np.abs(bdy_J_K2[1:-1] - bdy_J_K2[:-2]) / np.abs(bdy_J_K2[2:] - bdy_J_K2[1:-1])) / log10(bdy_r)]), axis=-1)

It seems we have **third-order convergence.**

In [ ]:
bdy_e_M_K1 = (bdy_M_K1[-2] - bdy_M_K1[-1]) / ((bdy_rr_infs[-1] / bdy_rr_infs[-2])**3 - 1.0)
bdy_e_M_K2 = (bdy_M_K2[-2] - bdy_M_K2[-1]) / ((bdy_rr_infs[-1] / bdy_rr_infs[-2])**3 - 1.0)
bdy_e_J_K1 = (bdy_J_K1[-2] - bdy_J_K1[-1]) / ((bdy_rr_infs[-1] / bdy_rr_infs[-2])**3 - 1.0)
bdy_e_J_K2 = (bdy_J_K2[-2] - bdy_J_K2[-1]) / ((bdy_rr_infs[-1] / bdy_rr_infs[-2])**3 - 1.0)

In [ ]:
print(bdy_e_M_K1, bdy_e_M_K2, bdy_e_J_K1, bdy_e_J_K2)

In [ ]:
print(bdy_e_M_K1 / bdy_M_K1[-1], bdy_e_M_K2 / bdy_M_K2[-1], bdy_e_J_K1 / bdy_J_K1[-1], bdy_e_J_K2 / bdy_J_K2[-1])

This allows us to calculate "better" expressions for the masses and angular momenta.

In [ ]:
bdy_f_M_K1 = bdy_M_K1[-1] - bdy_e_M_K1
bdy_f_M_K2 = bdy_M_K2[-1] - bdy_e_M_K2
bdy_f_J_K1 = bdy_J_K1[-1] - bdy_e_J_K1
bdy_f_J_K2 = bdy_J_K2[-1] - bdy_e_J_K2

Using these quantities we can present another set of graphs which again prove third order convergence in perhaps a more familiar way.

In [ ]:
fig, ax = plt.subplots(figsize = (8, 6))

ax.plot(np.log10(bdy_rr_infs), np.log10(np.abs(bdy_M_K1 - bdy_f_M_K1)), "-*", label = r"S($M_{Komar})$")
ax.plot(np.log10(bdy_rr_infs), np.log10(np.abs(bdy_M_K2 - bdy_f_M_K2)), "-*", label = r"V($M_{Komar})$")
ax.plot(np.log10(bdy_rr_infs), np.log10(np.abs(bdy_J_K1 - bdy_f_J_K1)), "-*", label = r"S($J_{Komar})$")
ax.plot(np.log10(bdy_rr_infs), np.log10(np.abs(bdy_J_K2 - bdy_f_J_K2)), "-*", label = r"V($J_{Komar})$")
ax.plot(np.log10(bdy_rr_infs), -3.0 * np.log10(bdy_rr_infs) + 3, "-", label = r"$\mathcal{O}\left(r_{bdy}^{-3}\right)$")

ax.legend()
ax.set_xlabel(r"$\log_{10}\,r_{bdy}$")
ax.set_ylabel(r"$\log_{10} \epsilon$")
ax.set_title("Komar Mass and Angular Momentum Boundary Errors")

plt.show()

Now check the error sizes.

In [ ]:
print(bdy_e_M_K1 / res_e_M_K1, bdy_e_M_K2 / res_e_M_K2, bdy_e_J_K1 / res_e_J_K1, bdy_e_J_K2 / res_e_J_K2)

This means that the boundary error is much greater than the resolution error.

# Komar Quantities Convergence Rate

In [ ]:
directory = "Boundary2/l=6,w=0.70000,dr=0.125,N=1024/"

In [ ]:
rr = np.genfromtxt(directory + "sph_rr.asc")[:,0]
th = np.genfromtxt(directory + "sph_th.asc")[0,:]

In [ ]:
M_K1 = np.genfromtxt(directory + "M_Komar1.asc")
M_K2 = np.genfromtxt(directory + "M_Komar2.asc")
J_K1 = np.genfromtxt(directory + "J_Komar1.asc")
J_K2 = np.genfromtxt(directory + "J_Komar2.asc")

In [ ]:
fig, ax = plt.subplots(figsize = (8, 6))
ax.plot(rr, M_K1)
ax.set_xlabel(r"$r$")
ax.set_ylabel(r"$M$")
ax.set_title(r"Komar Mass")

plt.show()

In [ ]:
fig, ax = plt.subplots(figsize = (8, 6))
ax.plot(rr, np.log10(np.abs(M_K1 - bdy_f_M_K1)), label = r"Boundary" )
ax.plot(rr, np.log10(np.abs(M_K1 - res_f_M_K1)), label = r"Resolution" )
ax.set_xlabel(r"$r$")
ax.set_ylabel(r"$\log_{10} \epsilon$")
ax.legend()
ax.set_title(r"Error in Komar Mass")
plt.show()